# Stage 3 ordinary FIT/DEV-S worker
CPU only. Run preparation once first. This worker publishes source statistics for G0, complete 25-head C blocks and immutable selection references. It does not enable ordinary SCREEN, DEV-C or CONFIRM scoring or accept Stage 3.

Start with MODE='profile' on each representative-owning partition. G0 separately owns seed0 C=.01 representatives. Before MODE='ordinary', supply indexes covering both true extremes and seed0 .01 for all six metadata-selected representatives. Retain those measurements and reuse the blocks. Completed units print their exact immutable references as they finish; capture them before interrupting.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage3-fits')
CODE_REF = 'phase1-stage3-fits'  # Resolved once to an immutable commit; never pull during a run.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Drazr/Research.git',str(REPO)],check=True)
if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git','fetch','origin',CODE_REF],cwd=REPO,check=True)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','FETCH_HEAD'],cwd=REPO,text=True).strip()
subprocess.run(['git','checkout','--detach',CODE_COMMIT],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements-stage2-cpu.txt')],check=True)
print('Fixed code commit:',CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
import json
MANIFEST = PHASE1_ROOT / 'REPLACE_WITH_PREPARED_STAGE3_PLAN/_COMPLETE.json'
FEATURE_SOURCE = 'vlm'  # vlm/e03. E03 runs once across both model comparisons.
MODEL = 'qwen'          # qwen/gemma for vlm; None for e03.
OWNER = 'qwen-cpu-1'
MODE = 'profile'        # profile/ordinary
# One shared, reviewed assignment file: {"owners": {"qwen-cpu-1": [group IDs], ...}}.
# It must cover all 145 group IDs exactly once. Each owner has one source/model.
OWNERSHIP_FILE = PHASE1_ROOT / 'REPLACE_WITH_EXPLICIT_OWNERSHIP.json'
RESUME_INDEXES = []     # Exact {"path": ..., "sha256": ...} mappings for this owner's ordinary units.
PROFILE_INDEXES = []    # Exact ordinary + G0 representative run-index refs, all six groups for bulk.


The ownership file is an input, never a mutable worker checkpoint. Preparation's plan contains the exact group IDs and profiling representatives. Agree on the full disjoint mapping before launching runtimes; never run two producers for the same group/assignment/C. Keep G0's assignments under preparation ownership. A statistics_ready event exposes that group's immutable statistics reference immediately for G0.

The child runs with one BLAS/OpenMP thread and streams JSON, including 15-second heartbeats, read/verify, fit/publication, semantic readback, fitted-head time, retries and process high-water RSS (Linux). Exclusive profiler read/hash/write components overlap enclosing phase elapsed times and must not be added twice. Keep the Colab output as the measurement record. Fixture timings cannot substitute for production measurements.


In [ ]:
if not MANIFEST.is_file() or not OWNERSHIP_FILE.is_file():
    raise RuntimeError('Set the explicit prepared plan and agreed ownership file after mounting Drive.')
if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()!=CODE_COMMIT or subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Checkout changed after setup.')
command = [sys.executable,'-u','-m','src.stage3.fit_readouts',
           '--phase1-root',str(PHASE1_ROOT),'--manifest',str(MANIFEST),
           '--ownership',str(OWNERSHIP_FILE),'--owner',OWNER,
           '--feature-source',FEATURE_SOURCE,'--mode',MODE,'--storage-scope','persistent-drive']
if MODEL is not None:
    command += ['--model',MODEL]
for flag, refs in [('--resume-index',RESUME_INDEXES),('--profile-index',PROFILE_INDEXES)]:
    for ref in refs:
        command += [flag,json.dumps(ref)]
process = subprocess.Popen(command,cwd=REPO,env=os.environ.copy(),stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT,text=True,bufsize=1)
try:
    for line in process.stdout:
        print(line,end='',flush=True)
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line,end='',flush=True)
    process.wait()
    raise
if process.wait()!=0:
    raise RuntimeError('Worker blocked/failed. Preserve streamed references and inspect the failure before resuming.')
